# Task 2 - RAG from scratch (no LangChain / LlamaIndex)
Documents: Llama 2 paper (arXiv 2307.09288, 77 pages) | Embeddings: BAAI/bge-small-en-v1.5 | Index: FAISS (exact cosine)
Evaluation: 15 hand-written questions, each with an exact evidence passage -> recall@k for 2 chunk sizes
Extras: BM25 from scratch, hybrid (RRF), cross-encoder re-ranker, final answer with Qwen2.5-1.5B-Instruct

In [ ]:
!pip -q install sentence-transformers faiss-cpu pypdf

In [ ]:
import re, json, bisect, unicodedata, os
from collections import Counter
import numpy as np, pandas as pd, requests, torch, faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

PDF_URL = "https://arxiv.org/pdf/2307.09288"          # any public PDF of 30+ pages works
EMB_MODEL = "BAAI/bge-small-en-v1.5"                    # 384-dim, max 512 tokens
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # bge expects this on queries only
CHUNK_CONFIGS = {"small_100w": (100, 20), "large_300w": (300, 50)}           # (words per chunk, overlap words)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("results", exist_ok=True)
print("device:", DEVICE)

In [ ]:
# 1) Load + clean the PDF, keep track of where each page starts
r = requests.get(PDF_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
r.raise_for_status()
open("doc.pdf", "wb").write(r.content)

def clean(t):
    t = unicodedata.normalize("NFKC", t)          # ligatures like "ﬁ" -> "fi"
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)        # re-join words hyphenated across line breaks
    return re.sub(r"\s+", " ", t).strip()         # collapse all whitespace

pages = [clean(p.extract_text() or "") for p in PdfReader("doc.pdf").pages]
full, page_starts = "", []
for p in pages:
    page_starts.append(len(full))
    full += p + " "
full_lower = full.lower()

def page_of(char_pos):                            # 1-indexed page number of a character offset
    return bisect.bisect_right(page_starts, char_pos)

print(len(pages), "pages |", len(full.split()), "words")

In [ ]:
# Helper for WRITING your questions: search the cleaned text and copy evidence from THIS output
def find(term, width=250, max_hits=5):
    hits = list(re.finditer(re.escape(term.lower()), full_lower))
    print(len(hits), "hits")
    for m in hits[:max_hits]:
        s = max(0, m.start() - width)
        print(f"[page {page_of(m.start())}] ...{full[s:m.end() + width]}...\n")

find("trillion tokens")      # example - try your own terms

In [ ]:
# 2) Chunking: fixed-size word windows with overlap; each chunk remembers its character span
words = [(m.start(), m.end()) for m in re.finditer(r"\S+", full)]

def make_chunks(size, overlap):
    step, out = size - overlap, []
    for i in range(0, len(words), step):
        w = words[i:i + size]
        s, e = w[0][0], w[-1][1]
        out.append({"start": s, "end": e, "page": page_of(s), "text": full[s:e]})
        if i + size >= len(words):
            break
    return out

In [ ]:
# 3) Embed every chunk and build one FAISS index per chunk size
emb = SentenceTransformer(EMB_MODEL, device=DEVICE)
INDEX = {}
for name, (size, ov) in CHUNK_CONFIGS.items():
    chunks = make_chunks(size, ov)
    X = emb.encode([c["text"] for c in chunks], batch_size=64, normalize_embeddings=True,
                   show_progress_bar=True).astype("float32")
    index = faiss.IndexFlatIP(X.shape[1])         # inner product on unit vectors == cosine similarity
    index.add(X)
    n_tok = [len(emb.tokenizer(c["text"])["input_ids"]) for c in chunks]
    INDEX[name] = {"chunks": chunks, "X": X, "index": index}
    print(f"{name}: {len(chunks)} chunks | avg {np.mean(n_tok):.0f} tokens | "
          f"{sum(t > emb.max_seq_length for t in n_tok)} truncated (limit {emb.max_seq_length})")

In [ ]:
# 4) YOUR 15 QUESTIONS - write them yourself.
# evidence = the exact sentence/phrase that answers the question, COPIED FROM find() OUTPUT (not from a PDF viewer).
# Keep evidence 10-40 words. Mix question types: direct facts, paraphrased (different words than the paper),
# numbers from tables, and a couple that need info spread over nearby sentences.
QUESTIONS = [
    # {"q": "Your question here?", "evidence": "exact text copied from find() output"},
]
assert len(QUESTIONS) >= 15, f"write at least 15 questions (have {len(QUESTIONS)})"

for qd in QUESTIONS:
    ev = clean(qd["evidence"]).lower()
    pos = full_lower.find(ev)
    assert pos != -1, f"Evidence not found - re-copy it from find(): {qd['evidence'][:70]}"
    qd["ev_start"], qd["ev_end"], qd["page"] = pos, pos + len(ev), page_of(pos)
json.dump(QUESTIONS, open("results/questions.json", "w"), indent=2)
print("all", len(QUESTIONS), "evidence passages located")

In [ ]:
# 5) Retrievers
def retrieve_dense(name, query, k=5):
    q = emb.encode([QUERY_PREFIX + query], normalize_embeddings=True).astype("float32")
    scores, ids = INDEX[name]["index"].search(q, k)
    return [int(i) for i in ids[0]], [float(s) for s in scores[0]]

def retrieve_numpy(name, query, k=5):             # same thing without FAISS, to show what FAISS does
    q = emb.encode([QUERY_PREFIX + query], normalize_embeddings=True)[0]
    s = INDEX[name]["X"] @ q
    ids = np.argsort(-s)[:k]
    return [int(i) for i in ids], [float(s[i]) for i in ids]

q0 = QUESTIONS[0]["q"]
assert retrieve_dense("small_100w", q0)[0] == retrieve_numpy("small_100w", q0)[0]
print("FAISS == NumPy brute force: OK")

In [ ]:
# 6) Evaluation: a retrieved chunk is a HIT if it contains >= 50% of the gold evidence span
def is_hit(chunk, qd):
    overlap = min(chunk["end"], qd["ev_end"]) - max(chunk["start"], qd["ev_start"])
    return overlap >= 0.5 * (qd["ev_end"] - qd["ev_start"])

def evaluate(name, retriever, depth=20):
    rows = []
    for qd in QUESTIONS:
        ids, _ = retriever(name, qd["q"], depth)
        ranks = [r for r, i in enumerate(ids, 1) if is_hit(INDEX[name]["chunks"][i], qd)]
        rows.append({"q": qd["q"], "gold_page": qd["page"], "rank": ranks[0] if ranks else None})
    df = pd.DataFrame(rows)
    res = {f"recall@{k}": float(((df["rank"].notna()) & (df["rank"] <= k)).mean()) for k in (1, 3, 5, 10)}
    res["MRR"] = float(df["rank"].apply(lambda r: 1 / r if pd.notna(r) else 0.0).mean())
    return res, df

results, per_q = {}, {}
for name in CHUNK_CONFIGS:
    results[(name, "dense")], per_q[(name, "dense")] = evaluate(name, retrieve_dense)
print(pd.DataFrame(results).T.round(3).to_string())

In [ ]:
# 7) Per-question ranks side by side (None = not in top 20)
side = pd.DataFrame({"q": [qd["q"] for qd in QUESTIONS], "gold_page": [qd["page"] for qd in QUESTIONS]})
for name in CHUNK_CONFIGS:
    side[name] = per_q[(name, "dense")]["rank"].values
pd.set_option("display.max_colwidth", 90)
print(side.to_string())

In [ ]:
# 8) Failure analysis: what came back instead of the gold passage?
def show_failures(name, method="dense", retriever=retrieve_dense, k=5, n_show=3):
    df = per_q[(name, method)]
    bad = df[df["rank"].isna() | (df["rank"] > k)]
    print(f"{len(bad)} questions missed in top-{k} for {name}/{method}\n")
    for idx in bad.index:
        qd = QUESTIONS[idx]
        print("Q:", qd["q"], "| gold page", qd["page"], "| gold rank:", df.loc[idx, "rank"])
        print("   GOLD:", qd["evidence"][:200])
        ids, scores = retriever(name, qd["q"], n_show)
        for i, s in zip(ids, scores):
            c = INDEX[name]["chunks"][i]
            print(f"   got p{c['page']} ({s:.3f}): {c['text'][:200]}...")
        print()

for name in CHUNK_CONFIGS:
    show_failures(name)

In [ ]:
# 9) [Optional, recommended] BM25 from scratch + hybrid search with Reciprocal Rank Fusion
def tokenize(t):
    return re.findall(r"[a-z0-9]+", t.lower())

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        toks = [tokenize(d) for d in docs]
        self.N = len(toks)
        self.len = np.array([len(t) for t in toks], dtype=float)
        self.avgdl = self.len.mean()
        self.tf = [Counter(t) for t in toks]
        df = Counter()
        for t in toks:
            df.update(set(t))
        self.idf = {w: np.log(1 + (self.N - n + 0.5) / (n + 0.5)) for w, n in df.items()}

    def scores(self, query):
        s = np.zeros(self.N)
        for w in set(tokenize(query)):
            if w not in self.idf:
                continue
            f = np.array([tf.get(w, 0) for tf in self.tf], dtype=float)
            s += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.len / self.avgdl))
        return s

for name in CHUNK_CONFIGS:
    INDEX[name]["bm25"] = BM25([c["text"] for c in INDEX[name]["chunks"]])

def retrieve_bm25(name, query, k=5):
    s = INDEX[name]["bm25"].scores(query)
    ids = np.argsort(-s)[:k]
    return [int(i) for i in ids], [float(s[i]) for i in ids]

def retrieve_hybrid(name, query, k=5, pool=50, c=60):
    fused = {}
    for ids in (retrieve_dense(name, query, pool)[0], retrieve_bm25(name, query, pool)[0]):
        for rank, i in enumerate(ids):
            fused[i] = fused.get(i, 0.0) + 1.0 / (c + rank + 1)
    top = sorted(fused, key=fused.get, reverse=True)[:k]
    return top, [fused[i] for i in top]

for name in CHUNK_CONFIGS:
    results[(name, "bm25")], per_q[(name, "bm25")] = evaluate(name, retrieve_bm25)
    results[(name, "hybrid")], per_q[(name, "hybrid")] = evaluate(name, retrieve_hybrid)
print(pd.DataFrame(results).T.round(3).to_string())

In [ ]:
# 10) [Optional, recommended] Cross-encoder re-ranker on top of hybrid candidates
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)

def retrieve_rerank(name, query, k=5, pool=20):
    ids, _ = retrieve_hybrid(name, query, pool)
    s = reranker.predict([(query, INDEX[name]["chunks"][i]["text"]) for i in ids])
    order = np.argsort(-s)[:k]
    return [ids[j] for j in order], [float(s[j]) for j in order]

for name in CHUNK_CONFIGS:
    results[(name, "hybrid+rerank")], per_q[(name, "hybrid+rerank")] = evaluate(name, retrieve_rerank)
table = pd.DataFrame(results).T.round(3)
print(table.to_string())
table.to_csv("results/retrieval_results.csv")

In [ ]:
# 11) Final answer generation with a small LLM, grounded in the top-5 chunks
from transformers import AutoTokenizer, AutoModelForCausalLM
LLM_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ltok = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE)

BEST = max(CHUNK_CONFIGS, key=lambda n: results[(n, "dense")]["recall@5"])   # change if you prefer another config

@torch.no_grad()
def answer(query, name=BEST, retriever=retrieve_dense, k=5):
    ids, _ = retriever(name, query, k)
    ctx = "\n\n".join(f"[{j + 1}] (page {INDEX[name]['chunks'][i]['page']}) {INDEX[name]['chunks'][i]['text']}"
                      for j, i in enumerate(ids))
    msgs = [{"role": "system", "content": "Answer the question using ONLY the context passages. Cite passages like [1]. "
                                          "If the answer is not in the context, say \"I don't know\"."},
            {"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {query}"}]
    enc = ltok(ltok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False), return_tensors="pt").to(DEVICE)
    out = llm.generate(**enc, max_new_tokens=200, do_sample=False)
    return ltok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

answers = []
for qd in QUESTIONS:
    a = answer(qd["q"])
    answers.append({"q": qd["q"], "answer": a, "gold_evidence": qd["evidence"]})
    print("Q:", qd["q"], "\nA:", a, "\n")
pd.DataFrame(answers).to_csv("results/answers.csv", index=False)

In [ ]:
# 12) Save + download
side.to_csv("results/per_question_ranks_dense.csv", index=False)
json.dump({f"{n}|{m}": v for (n, m), v in results.items()}, open("results/metrics.json", "w"), indent=2)
!zip -qr results_task2.zip results
try:
    from google.colab import files
    files.download("results_task2.zip")
except ImportError:
    print("results_task2.zip saved in the working directory")